In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os, sys 

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
mart_schema = get_mart_schema() 

from superstore_logger import get_superstore_logger, log_event
from superstore_backfill_utils import is_dry_run
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_mart_product_performance = get_superstore_logger("superstore_mart_product_performance")


# -----------------------------
# Dry run
# -----------------------------
# The serving layer rebuilds from Gold and has no run-mode logic of its own —
# a replay or backfill simply produces different Gold for it to read. But it
# does write, so it must not write during a dry run. is_dry_run reads only the
# dry_run parameter and never raises, so a serving task cannot fail on the
# full_refresh gate that is enforced upstream where it belongs.
if is_dry_run(dbutils):
    log_event(
        logger_mart_product_performance,
        "INFO",
        "Dry run: skipping mart_production_performance write",
        dry_run=True,
        layer="Mart"
    )
    dbutils.notebook.exit("DRY_RUN_COMPLETED layer=mart_production_performance (no data written)")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_mart_product_performance,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="Feature Engineering"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
required_tables = [
    f"{catalog}.{gold_schema}.dim_customers",
    f"{catalog}.{gold_schema}.facts_orders",
    f"{catalog}.{gold_schema}.facts_sales"
]

missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, gold_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_mart_product_performance, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create mart production performance table.", layer="Mart")
else:
  # -------------------------------
  # Create mart Table (Environment-aware)
  # -------------------------------
  spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.{mart_schema}.mart_product_performance AS
  WITH product_sales AS (
    SELECT
      p.product_id,
      p.product_name,
      p.category,
      p.sub_category,
      SUM(s.sales) AS total_sales,
      SUM(s.profit) AS total_profit,
      SUM(s.quantity) AS total_quantity,
      AVG(s.discount) AS avg_discount,
      MIN(o.order_date) AS first_order_date,
      MAX(o.order_date) AS last_order_date
    FROM {catalog}.{gold_schema}.dim_products p
    LEFT JOIN {catalog}.{gold_schema}.facts_sales s
      ON p.product_id = s.product_id
    LEFT JOIN {catalog}.{gold_schema}.facts_orders o
      ON s.order_id = o.order_id
    WHERE p.is_current = true
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
  )

  SELECT
    product_id,
    product_name,
    category,
    sub_category,
    total_sales,
    total_profit,
    total_quantity,
    avg_discount,
    CASE 
    WHEN total_sales = 0 THEN NULL 
    ELSE ROUND(total_profit / total_sales, 4) 
    END AS profit_margin,
    CASE 
      WHEN DATEDIFF(last_order_date, first_order_date) = 0 THEN total_quantity 
      ELSE ROUND(total_quantity / DATEDIFF(last_order_date, first_order_date), 2) 
    END AS sales_velocity,
    CASE 
    WHEN total_profit < 0 THEN 1 
    ELSE 0 
    END AS loss_making_flag
  FROM product_sales;
  """)

log_event(logger_mart_product_performance, "INFO", "Mart product performance table created successfully.", layer="Mart")


